In [8]:
%pip install pymongo
%pip install psycopg2

from jsonschema import validate
import pytest
import json
import sys
from pathlib import Path
from unittest.mock import patch, MagicMock
search_paths = [Path.cwd(), *Path.cwd().parents]
for search_path in search_paths:
    if (search_path / "app").exists():
        sys.path.insert(0, str(search_path))
        break

from app.services.ai_service import ai_service
from app.models.enums import WalletType
from app.schemas.wallet import WalletCreate
from app.services.wallet_service import wallet_service

# Class giả lập lại cấu trúc Response của Google Gemini API
class MockGeminiResponse:
    def __init__(self, text):
        self.text = text

# ===============================================
# TC-10: AI Trích xuất giao dịch (Natural Language Parsing)
# ===============================================
def test_parse_natural_language(db_session, test_user, test_category):
    # Cần tạo ít nhất 1 ví để vượt qua logic check trong UserService
    w1 = wallet_service.create(db_session, WalletCreate(name="Cash", type=WalletType.cash, initial_balance=0, currency="VND"), test_user.user_id)

    mock_json_bot_talks = """
    ```json
    {
        "transactions": [
            {
                "amount": 50000,
                "transaction_type": "expense",
                "category_id": 1,
                "wallet_id": 1,
                "note": "Ăn phở"
            }
        ]
    }
    ```
    """
    # Xài MagicMock để "bịt mắt" hàm gọi Google Gemini Internet
    with patch('app.services.ai_service.AIService._get_model') as mock_get_model:
        mock_model_instance = MagicMock()
        mock_model_instance.generate_content.return_value = MockGeminiResponse(mock_json_bot_talks)
        mock_get_model.return_value = mock_model_instance
        
        res = ai_service.parse_natural_language(db_session, "sáng nay ăn phở 50k", test_user.user_id)
        
        # Test kiểm chứng AI Service lấy ra Json chuẩn
        assert "transactions" in res
        assert len(res["transactions"]) == 1
        assert res["transactions"][0]["amount"] == 50000
        assert res["transactions"][0]["note"] == "Ăn phở"

# ===============================================
# TC-11: Chatbot RAG (Thêm giao dịch via Chat)  
# ===============================================
def test_chat_rag(db_session, test_user):
    mock_chat_function_calling_response = """Tuyệt vời, mình giúp bạn thêm nha
```json
{
    "action": "add_transaction",
    "data": [
        {
            "amount": 30000,
            "transaction_type": "expense",
            "category_id": 1,
            "wallet_id": 1,
            "note": "Cà phê"
        }
    ],
    "reply": "Đã thêm giao dịch vào hệ thống."
}
```"""
    
    # Ở đây hàm chat_rag sẽ chép Logs vào MongoDB. Ta phải bịt MongoDB lại!
    with patch('app.services.ai_service.chat_collection') as mock_mongo:
        mock_mongo.insert_one = MagicMock()
        mock_mongo.find.return_value.sort.return_value.limit.return_value = [] # Giả sử chưa có lịch sử chat
        
        with patch('app.services.ai_service.AIService._get_model') as mock_get_model:
            mock_model_instance = MagicMock()
            mock_model_instance.generate_content.return_value = MockGeminiResponse(mock_chat_function_calling_response)
            mock_get_model.return_value = mock_model_instance

            # Gọi RAG với câu lệnh tự nhiên
            res = ai_service.chat_rag(db_session, "Nhớ thêm 30k cafe hôm nay", "req-123", test_user.user_id)
            
            assert res["action"] == "add_transaction"
            assert len(res["action_data"]) == 1
            assert res["action_data"][0]["amount"] == 30000
            assert res["action_data"][0]["note"] == "Cà phê"
            assert res["session_id"] == "req-123"

# ===============================================
# TC-12: OCR (Đọc File ảnh biên lai)
# ===============================================
def test_ocr_receipt():
    # File fake return format
    mock_ocr_response = """```json
{
    "merchant": "Highlands Coffee",
    "total": 59000,
    "date": "2026-03-27",
    "items": []
}
```"""
    # Ngăn việc dùng pillow mở ảnh lên báo lỗi byte ảo
    with patch('app.services.ai_service.Image.open'): 
        with patch('app.services.ai_service.AIService._get_model') as mock_get_model:
            mock_model_instance = MagicMock()
            mock_model_instance.generate_content.return_value = MockGeminiResponse(mock_ocr_response)
            mock_get_model.return_value = mock_model_instance

            res = ai_service.ocr_receipt(b"fake_image_bytes")
            assert res["merchant"] == "Highlands Coffee"
            assert res["total"] == 59000

# ===============================================
# TC-13: AI Auto-Generate Budget
# ===============================================
def test_generate_budget_template(db_session, test_user):
    mock_budget_template_res = """
    [
        {"category_id": 1, "category_name": "Ăn uống", "amount_limit": 5000000, "description": "Needs 50%"},
        {"category_id": 2, "category_name": "Giải trí", "amount_limit": 3000000, "description": "Wants 30%"},
        {"category_id": 3, "category_name": "Tiết kiệm", "amount_limit": 2000000, "description": "Savings 20%"}
    ]
    """
    with patch('app.services.ai_service.AIService._get_model') as mock_get_model:
        mock_model_instance = MagicMock()
        mock_model_instance.generate_content.return_value = MockGeminiResponse(mock_budget_template_res)
        mock_get_model.return_value = mock_model_instance

        # User thu nhập 10 củ, muốn dùng rule 50/30/20
        res = ai_service.generate_budget_template(db_session, income=10000000, template_type="50/30/20", user_id=test_user.user_id)
        
        assert len(res) == 3
        assert res[0]["amount_limit"] == 5000000
        assert res[2]["amount_limit"] == 2000000
        assert "50/30/20" in res[1]["description"] or "30%" in res[1]["description"]


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install email-validator
%pip install python-jose
%pip install passlib
import pytest
from app.services.auth_service import auth_service
from app.schemas.user import UserCreate

# ===============================================
# TC-14: Đăng ký tài khoản (User Registration)
# ===============================================
def test_user_registration(db_session):
    # HP: Đăng ký thành công
    user_in = UserCreate(
        username="newuser_123",
        email="newuser@example.com",
        password="SecurePassword123!",
        full_name="New Test User"
    )
    new_user = auth_service.register(db_session, user_in)
    
    assert new_user.username == "newuser_123"
    assert new_user.email == "newuser@example.com"
    assert new_user.is_active is True
    # Đảm bảo password đã được băm chứ không lưu plaintext
    assert new_user.password_hash != "SecurePassword123!"

    # UP: Đăng ký với Email đã tồn tại (Phải bị văng Exception)
    with pytest.raises(ValueError, match="Email này đã được đăng ký."):
        duplicate_email_user = UserCreate(
            username="another_user",
            email="newuser@example.com",
            password="Password111",
            full_name="Duplicate"
        )
        auth_service.register(db_session, duplicate_email_user)

    # UP: Đăng ký với Username đã tồn tại
    with pytest.raises(ValueError, match="Username này đã tồn tại."):
        duplicate_username_user = UserCreate(
            username="newuser_123",
            email="different@example.com",
            password="Password222",
            full_name="Duplicate"
        )
        auth_service.register(db_session, duplicate_username_user)


# ===============================================
# TC-15: Đăng nhập & Cơ chế Token (JWT Authentication)
# ===============================================
def test_user_login_and_token_refresh(db_session):
    # Chuẩn bị dữ liệu: Tạo User trước
    user_in = UserCreate(username="login_test", email="login@test.com", password="MyPassword1!", full_name="Test")
    auth_service.register(db_session, user_in)

    # HP: Đăng nhập thành công với Username
    tokens = auth_service.login(db_session, identity="login_test", password="MyPassword1!")
    assert "access_token" in tokens
    assert "refresh_token" in tokens
    assert tokens["token_type"] == "bearer"

    # HP: Đăng nhập thành công với Email
    tokens_by_email = auth_service.login(db_session, identity="login@test.com", password="MyPassword1!")
    assert "access_token" in tokens_by_email

    # UP: Sai mật khẩu
    with pytest.raises(ValueError, match="Sai tài khoản hoặc mật khẩu"):
        auth_service.login(db_session, identity="login_test", password="WrongPassword")
        
    # UP: Sai tài khoản
    with pytest.raises(ValueError, match="Sai tài khoản hoặc mật khẩu"):
        auth_service.login(db_session, identity="idontexist", password="MyPassword1!")

    # =============================================
    # Giao thức Refresh Token
    # =============================================
    refresh_token = tokens["refresh_token"]
    
    # Lấy access_token mới bằng refresh_token hợp lệ
    new_tokens = auth_service.refresh_token(db_session, refresh_token=refresh_token)
    assert new_tokens["access_token"] != tokens["access_token"]

    # =============================================
    # Giao thức Logout (Blacklist token)
    # =============================================
    auth_service.logout(db_session, refresh_token=refresh_token)
    
    # Cố tình dùng token đã bị logout để lấy quyền
    with pytest.raises(ValueError, match="Token đã bị thu hồi \\(Người dùng đã đăng xuất\\)"):
        auth_service.refresh_token(db_session, refresh_token=refresh_token)


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: email-validator in c:\users\minht\appdata\local\programs\python\python310\lib\site-packages (2.3.0)




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pytest
from datetime import date
from decimal import Decimal

from app.models.enums import WalletType, DebtType, InvestmentType
from app.schemas.wallet import WalletCreate
from app.schemas.debt import DebtCreate, DebtRepaymentCreate
from app.schemas.investment import InvestmentCreate, InvestmentSell, InvestmentPassiveIncome

from app.services.wallet_service import wallet_service
from app.services.debt_service import debt_service
from app.services.investment_service import investment_service

# ===============================================
# TC-06: Tạo khoản Nợ (Receivable - Cho vay)
# ===============================================
def test_create_receivable_debt(db_session, test_user, test_category):
    # Khởi tạo ví tiền mặt có 100k
    w_cash = wallet_service.create(db_session, WalletCreate(name="Cash", type=WalletType.cash, initial_balance=100000, currency="VND"), test_user.user_id)

    # HP: Cho anh Tuấn vay 30k
    debt_in = DebtCreate(
        creditor_name="Anh Tuấn", 
        type=DebtType.receivable, 
        total_amount=30000, 
        wallet_id=w_cash.wallet_id, 
        category_id=test_category.category_id,
        due_date=date.today()
    )
    new_debt = debt_service.create_debt(db_session, debt_in, test_user.user_id)

    # Kì vọng: Ví tiền mặt bị giảm 30k (còn 70k)
    assert new_debt.remaining_amount == 30000
    assert w_cash.balance == 70000

    # UP: Thử cho vay 200k (ví không đủ tiền)
    with pytest.raises(ValueError, match="Ví không đủ số dư"):
        debt_in_fail = DebtCreate(
            creditor_name="Anh Bình", 
            type=DebtType.receivable, 
            total_amount=200000, 
            wallet_id=w_cash.wallet_id, 
            category_id=test_category.category_id
        )
        debt_service.create_debt(db_session, debt_in_fail, test_user.user_id)

# ===============================================
# TC-07: Trả Nợ / Thu Nợ (Repayment)
# ===============================================
def test_repay_debt(db_session, test_user, test_category):
    # Khởi tạo ví và khoản nợ (Vay của ngân hàng 50k - Payable)
    w_bank = wallet_service.create(db_session, WalletCreate(name="Bank", type=WalletType.bank, initial_balance=20000, currency="VND"), test_user.user_id)
    
    # Việc tạo Payable (đi vay) sẽ CỘNG tiền vào ví Bank. (Số dư mới ví Bank: 20k + 50k = 70k)
    debt_in = DebtCreate(
        creditor_name="Ngân hàng", 
        type=DebtType.payable, 
        total_amount=50000, 
        wallet_id=w_bank.wallet_id, 
        category_id=test_category.category_id
    )
    new_debt = debt_service.create_debt(db_session, debt_in, test_user.user_id)
    assert w_bank.balance == 70000
    assert new_debt.remaining_amount == 50000

    # HP: Trả ngân hàng 10k -> Trừ ví 10k, nợ giảm còn 40k
    repay_in = DebtRepaymentCreate(
        amount=10000,
        wallet_id=w_bank.wallet_id,
        category_id=test_category.category_id,
        date=date.today(),
        note="Trả góp đợt 1"
    )
    debt_service.repay_debt(db_session, new_debt.debt_id, repay_in, test_user.user_id)
    assert w_bank.balance == 60000
    assert new_debt.remaining_amount == 40000

    # UP: Trả 60k cho khoản nợ chỉ còn 40k -> Chặn
    with pytest.raises(ValueError, match="Số tiền trả không được lớn hơn số nợ còn lại"):
        repay_up = DebtRepaymentCreate(amount=60000, wallet_id=w_bank.wallet_id, category_id=test_category.category_id, date=date.today())
        debt_service.repay_debt(db_session, new_debt.debt_id, repay_up, test_user.user_id)

# ===============================================
# TC-08 & TC-09: Đầu tư (Mở, Passive Income, Bán Chốt lời)
# ===============================================
def test_investment_lifecycle(db_session, test_user):
    w_cash = wallet_service.create(db_session, WalletCreate(name="Cash", type=WalletType.cash, initial_balance=500000, currency="VND"), test_user.user_id)
    
    # 1. Mua Vàng SJC (TC-08)
    # Vốn = 100k, phí + thuế = 10k -> Phải trừ ví 110k (còn 390k)
    inv_in = InvestmentCreate(
        wallet_id=w_cash.wallet_id,
        name="Vàng SJC",
        type=InvestmentType.gold,
        quantity=Decimal('2.5'),
        principal_amount=100000,
        fee=5000,
        tax=5000
    )
    new_inv = investment_service.create_investment(db_session, inv_in, test_user.user_id)
    assert new_inv.principal_amount == 100000
    assert new_inv.quantity == Decimal('2.5')
    assert w_cash.balance == 390000

    # 2. Nhận cổ tức tiền mặt (TC-09)
    # Nhận 5k cổ tức -> Ví tăng 5k (lên 395k)
    inc_in = InvestmentPassiveIncome(amount=5000, wallet_id=w_cash.wallet_id)
    investment_service.receive_passive_income(db_session, new_inv.investment_id, inc_in, test_user.user_id)
    assert w_cash.balance == 395000

    # 3. Bán Chốt Lời / Lỗ (TC-09)
    # Bán giá 120k, phí = 5k. Lợi nhuận thực tế (Pnl = Selling Price - Principal - Tax - Fee - BuyTax - BuyFee). 
    # Nhưng trong logic thiết kế: PnL = selling_price - principal_amount - fee - tax. => 120k - 100k - 5k - 0 = +15k.
    # Tiền thực thu về ví: selling_price - fee - tax = 120k - 5k = 115k
    # => Ví mới: 395k + 115k = 510k.
    sell_in = InvestmentSell(
        selling_price=120000, 
        wallet_id=w_cash.wallet_id, 
        fee=5000, 
        tax=0, 
        date=date.today()
    )
    result = investment_service.sell(db_session, new_inv.investment_id, sell_in, test_user.user_id)
    
    assert result["profit"] == 15000
    assert w_cash.balance == 510000
    # Đã bán thì không được nhận thêm passive income nữa (Tài sản bị soft/hard delete nên báo không tìm thấy)
    with pytest.raises(ValueError, match="Không tìm thấy tài sản"):
        investment_service.receive_passive_income(db_session, new_inv.investment_id, inc_in, test_user.user_id)


In [11]:
import pytest
from datetime import date
from decimal import Decimal
from app.models.user import User
from app.models.wallet_category import Category
from app.models.enums import WalletType, TransactionType
from app.schemas.wallet import WalletCreate
from app.schemas.transaction import TransactionCreate, TransactionTransfer, TransactionUpdate
from app.services.wallet_service import wallet_service
from app.services.transaction_service import transaction_service
from app.core.security import get_password_hash

# ===============================================
# TC-01: Khởi tạo ví tiền mặt và thẻ tín dụng
# ===============================================
def test_create_wallets(db_session, test_user):
    # Mở tài khoản tiền mặt (có số dư khởi tạo)
    wallet_in = WalletCreate(name="Ví tiền mặt", type=WalletType.cash, initial_balance=10000, currency="VND")
    new_wallet = wallet_service.create(db_session, wallet_in, test_user.user_id)
    assert new_wallet.balance == 10000
    assert new_wallet.type == WalletType.cash

    # Mở thẻ tín dụng (Bắt đầu với 0 đồng, có hạn mức)
    credit_in = WalletCreate(name="Thẻ VISA", type=WalletType.credit, initial_balance=0, credit_limit=50000, currency="VND")
    new_credit = wallet_service.create(db_session, credit_in, test_user.user_id)
    assert new_credit.balance == 0
    assert new_credit.type == WalletType.credit

# ===============================================
# TC-02: Chi tiêu bằng ví Tiền mặt (Kiểm tra Safe-guard)
# ===============================================
def test_spend_cash_wallet(db_session, test_user, test_category):
    w1 = wallet_service.create(db_session, WalletCreate(name="Cash", type=WalletType.cash, initial_balance=50000, currency="VND"), test_user.user_id)

    # HP: Chi tiêu hợp lệ (ít hơn số tiền đang có)
    tx_in = TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=20000, transaction_type=TransactionType.expense, date=date.today())
    transaction_service.create(db_session, tx_in, test_user.user_id)
    assert w1.balance == 30000

    # UP: Xài lố số tiền mặt trong túi (vượt 30k) -> Chặn lại
    with pytest.raises(ValueError, match="Ví nguồn không đủ số dư để chi tiêu"):
        tx_in2 = TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=40000, transaction_type=TransactionType.expense, date=date.today())
        transaction_service.create(db_session, tx_in2, test_user.user_id)

# ===============================================
# TC-03: Chi tiêu bằng ví Thẻ tín dụng (Âm tiền ròng tự do)
# ===============================================
def test_spend_credit_wallet(db_session, test_user, test_category):
    w2 = wallet_service.create(db_session, WalletCreate(name="Credit", type=WalletType.credit, initial_balance=0, credit_limit=50000, currency="VND"), test_user.user_id)

    # HP: Thẻ tín dụng cho phép tiêu đến "âm tài khoản"
    tx_in = TransactionCreate(wallet_id=w2.wallet_id, category_id=test_category.category_id, amount=80000, transaction_type=TransactionType.expense, date=date.today())
    transaction_service.create(db_session, tx_in, test_user.user_id)
    assert w2.balance == Decimal('-80000')

# ===============================================
# TC-04: Tính chất Atomic - Chuyển giao tiền bạc
# ===============================================
def test_transfer_wallets(db_session, test_user):
    w1 = wallet_service.create(db_session, WalletCreate(name="Bank1", type=WalletType.bank, initial_balance=100000, currency="VND"), test_user.user_id)
    w2 = wallet_service.create(db_session, WalletCreate(name="Bank2", type=WalletType.bank, initial_balance=0, currency="VND"), test_user.user_id)

    # HP: Chuyển 20k thành công
    t_in = TransactionTransfer(source_wallet_id=w1.wallet_id, dest_wallet_id=w2.wallet_id, amount=20000, date=date.today())
    transaction_service.transfer(db_session, t_in, test_user.user_id)
    
    assert w1.balance == 80000
    assert w2.balance == 20000

    # UP: Ngăn chặn chuyển số tiền ma (Vượt quá Bank 1)
    with pytest.raises(ValueError, match="Ví nguồn không đủ số dư"):
        t_in2 = TransactionTransfer(source_wallet_id=w1.wallet_id, dest_wallet_id=w2.wallet_id, amount=90000, date=date.today())
        transaction_service.transfer(db_session, t_in2, test_user.user_id)

# ===============================================
# TC-05: Sửa & Xoá giao dịch làm thay đổi Wallet Balance
# ===============================================
def test_update_and_delete_transaction(db_session, test_user, test_category):
    w1 = wallet_service.create(db_session, WalletCreate(name="Cash", type=WalletType.cash, initial_balance=50000, currency="VND"), test_user.user_id)
    
    # User tiêu 10.000 -> Còn 40.000
    tx_in = TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=10000, transaction_type=TransactionType.expense, date=date.today())
    tx = transaction_service.create(db_session, tx_in, test_user.user_id)
    assert w1.balance == 40000

    # Sửa số tiền bill từ 10.000 thành 30.000 -> Số dư ví tiếp tục trừ thêm 20k
    tx_in_update = TransactionUpdate(amount=30000)
    transaction_service.update(db_session, tx.transaction_id, tx_in_update, test_user.user_id)
    assert w1.balance == 20000

    # Hệ thống hủy Bill hoàn toàn -> Số dư tự trả lại 30.000 về như cũ
    transaction_service.delete(db_session, tx.transaction_id, test_user.user_id)
    assert w1.balance == 50000

# ===============================================
# TC-06: Khớp lệnh Nhập liệu Hàng loạt (Bulk Insert)
# ===============================================
def test_bulk_insert_transactions(db_session, test_user, test_category):
    w1 = wallet_service.create(db_session, WalletCreate(name="Salary Bank", type=WalletType.bank, initial_balance=100000, currency="VND"), test_user.user_id)
    
    tx_list = [
        TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=20000, transaction_type=TransactionType.expense, date=date.today(), note="Tx 1"),
        TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=30000, transaction_type=TransactionType.expense, date=date.today(), note="Tx 2"),
        TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=10000, transaction_type=TransactionType.income, date=date.today(), note="Tx 3")
    ]
    
    # Bơm 1 chuỗi 3 giao dịch
    inserted_count = transaction_service.create_bulk(db_session, tx_list, test_user.user_id)
    assert inserted_count == 3
    
    # Kiểm chứng số dư cuối cùng được cộng dồn đúng 100k - 20k - 30k + 10k = 60000
    assert w1.balance == 60000
    
    # Khi 1 dòng (Chi phí lớn) làm âm ví -> Được phép vượt rào mặc định ở chế độ Bulk
    tx_list_over = [
        TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=70000, transaction_type=TransactionType.expense, date=date.today(), note="Nhập lố")
    ]
    inserted_over = transaction_service.create_bulk(db_session, tx_list_over, test_user.user_id)
    assert inserted_over == 1
    assert w1.balance == -10000  # Số dư có thể rớt xuống âm do Bulk Import được bypass an toàn
    
    # Nếu Test tường minh cờ bypass = False
    tx_list_block = [
        TransactionCreate(wallet_id=w1.wallet_id, category_id=test_category.category_id, amount=50000, transaction_type=TransactionType.expense, date=date.today(), note="Bị block")
    ]
    with pytest.raises(ValueError, match="không đủ tiền cho khoản chi"):
        transaction_service.create_bulk(db_session, tx_list_block, test_user.user_id, ignore_spend_limit=False)

# ===============================================
# TC-07: Tự động khởi tạo Ví, Danh mục và Khoản nợ từ Bulk Import
# ===============================================
def test_bulk_insert_auto_generation(db_session, test_user):
    from app.models.wallet_category import Category, Wallet
    from app.models.finance_modules import Debt, DebtRepayment
    
    tx_list = [
        # Giao dịch 1: Khởi tạo Ví "Ví Tiết Kiệm" và Danh mục "Lương thưởng"
        TransactionCreate(
            wallet_id=-1, new_wallet_name="Ví Tiết Kiệm",
            category_id=-1, new_category_name="Lương thưởng",
            amount=5000000, transaction_type=TransactionType.income, date=date.today(), note="Lương tháng mới"
        ),
        # Giao dịch 2: Sinh khoản Cho Vay (Nợ) mới do Danh mục "Cho vay"
        TransactionCreate(
            wallet_id=-1, new_wallet_name="Ví Tiết Kiệm", # Map lại ví đã sinh tạm ở Cache
            category_id=-2, new_category_name="Khác - Cho vay",
            amount=1000000, transaction_type=TransactionType.expense, date=date.today(), 
            note="Cho anh Sơn mượn", creditor_name="Sơn"
        )
    ]
    
    inserted_count = transaction_service.create_bulk(db_session, tx_list, test_user.user_id)
    assert inserted_count == 2
    
    # 1. Xác minh Wallet được tạo tự động
    new_wallet = db_session.query(Wallet).filter(Wallet.user_id == test_user.user_id, Wallet.name == "Ví Tiết Kiệm").first()
    assert new_wallet is not None
    assert new_wallet.balance == 4000000  # 5m - 1m
    
    # 2. Xác minh Category
    cat_salary = db_session.query(Category).filter(Category.name == "Lương thưởng").first()
    assert cat_salary is not None
    
    # 3. Xác minh Nợ (Debt)
    debt = db_session.query(Debt).filter(Debt.creditor_name == "Sơn").first()
    assert debt is not None
    assert debt.total_amount == 1000000
    assert debt.remaining_amount == 1000000
    
    # 4. Xác minh Gạch nợ
    tx_repay = [
        TransactionCreate(
            wallet_id=new_wallet.wallet_id,
            category_id=-3, new_category_name="Khác - Thu nợ",
            amount=500000, transaction_type=TransactionType.income, date=date.today(),
            note="Sơn trả mượn", creditor_name="Sơn"
        )
    ]
    transaction_service.create_bulk(db_session, tx_repay, test_user.user_id)
    
    db_session.refresh(debt)
    db_session.refresh(new_wallet)
    assert debt.remaining_amount == 500000
    assert new_wallet.balance == 4500000
